In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import cluster

import numpy as np
import matplotlib.pyplot as plt
import random

- Clustering is just another optimization problem, we are minimazing variability (c), that is the sum of the distance between the mean of the cluster and each example of the cluster (squared)

- Or/and  dissimilarity  that is the variability of the clusters

- Interesting how we are not dividing by n, the number of elements in the clusters, why? Because that would normalize the metric! By not normalizing we are saying that we want to penalize big diverse clusters more than small highly diverse clusters. Big and bad is worst that small and bad

- We could say that the optmization problem that we want to solve is to find the nuber of clusters that minize the similarity... but there is a foolish descision that would find the best value for that... that the **number of clusters would be equal to the number of values!!** and of course that is not the point, we want to find the best clusters to respresnt our data!

- So we need a constraint! And one of the best and simplest is to constrain the number of clustesrs! **We want to have only N clusters!**


## Hierchical clustering

1. Start by assigning each item to a cluster, so that if we have  N items, we now have N clusters, each containing only 1

2. Find the closest (most similar) pair of clusters and merge them into a single cluster and we'll have fewer clusters

3. Continue the p rocess unitl all items are clustered into a single cluster   of size N

# Examples

In [21]:
class Patient(cluster.Example):
    pass


def scaleAttrs(vals):
    vals = np.array(vals)
    mean = sum(vals) / len(vals)

    sd = np.std(vals)

    vals = vals - mean

    return vals / sd


def getData(toScale=False):

    hrList, stElevList, ageList, prevACSList, classList = [], [], [], [], []

    cardiacData = open("../data/cardiacData.txt", "r")

    for l in cardiacData:
        l = l.split(",")

        hrList.append(int(l[0]))
        stElevList.append(int(l[1]))
        ageList.append(int(l[2]))
        prevACSList.append(int(l[3]))
        classList.append(int(l[4]))

    if toScale:
        hrList = scaleAttrs(hrList)
        stElevList = scaleAttrs(stElevList)
        ageList = scaleAttrs(ageList)
        prevACSList = scaleAttrs(prevACSList)

    points = []
    for i in range(len(hrList)):
        features = np.array([hrList[i], prevACSList[i], stElevList[i], ageList[i]])
        pIndex = str(i)
        points.append(Patient("P" + pIndex, features, classList[i]))

    return points


def kmeans(examples, k, verbose=False):
    # Get k randomly chosen initial centroids, create cluster for each
    initialCentroids = random.sample(examples, k)
    clusters = []
    for e in initialCentroids:
        clusters.append(cluster.Cluster([e]))

    # Iterate until centroids do not change
    converged = False
    numIterations = 0
    while not converged:
        numIterations += 1
        # Create a list containing k distinct empty lists
        newClusters = []
        for i in range(k):
            newClusters.append([])

        # Associate each example with closest centroid
        for e in examples:
            # Find the centroid closest to e
            smallestDistance = e.distance(clusters[0].getCentroid())
            index = 0
            for i in range(1, k):
                distance = e.distance(clusters[i].getCentroid())
                if distance < smallestDistance:
                    smallestDistance = distance
                    index = i
            # Add e to the list of examples for appropriate cluster
            newClusters[index].append(e)

        for c in newClusters:  # Avoid having empty clusters
            if len(c) == 0:
                raise ValueError("Empty Cluster")

        # Update each cluster; check if a centroid has changed
        converged = True
        for i in range(k):
            if clusters[i].update(newClusters[i]) > 0.0:
                converged = False
        if verbose:
            print("Iteration #" + str(numIterations))
            for c in clusters:
                print(c)
            print("")
    return clusters


def trykmeans(examples, numClusters, numTrials, verbose=False):

    best = kmeans(examples, numClusters, verbose)
    minDissimilarity = cluster.dissimilarity(best)
    trial = 1
    while trial < numTrials:
        try:
            clusters = kmeans(examples, numClusters, verbose)

        except ValueError:
            continue

        currDissimilarity = cluster.dissimilarity(clusters)
        if currDissimilarity < minDissimilarity:
            best = clusters
            minDissimilarity = currDissimilarity

        trial += 1

    return best


def printClustering(clustering):
    """Assumes: clustering is a sequence of clusters
    Prints information about each cluster
    Returns list of fraction of pos cases in each cluster"""
    posFracs = []
    for c in clustering:
        numPts = 0
        numPos = 0
        for p in c.members():
            numPts += 1
            if p.getLabel() == 1:
                numPos += 1
        fracPos = numPos / numPts
        posFracs.append(fracPos)
        print(
            "Cluster of size", numPts, "with fraction of positives =", round(fracPos, 4)
        )
    return np.array(posFracs)


def testClustering(patients, numClusters, seed=0, numTrials=5):
    random.seed(seed)
    bestClustering = trykmeans(patients, numClusters, numTrials)
    posFracs = printClustering(bestClustering)
    return posFracs

In [22]:
patients = getData()
for k in (2,):
    print(" \n Test -kmeans (k = " + str(k) + ")")
    posFracs = testClustering(patients, k, 2)

 
 Test -kmeans (k = 2)
Cluster of size 118 with fraction of positives = 0.3305
Cluster of size 132 with fraction of positives = 0.3333


In [23]:
numPos = 0
for p in patients:
    if p.getLabel() == 1:
        numPos += 1

print("Total number of positive patients:", numPos)

Total number of positive patients: 83
